## Automated Data Acquisition & 3D Structure Generation
The first step in a fully automated, code-based molecular docking pipeline. It handles the bulk acquisition of biological macromolecules (proteins) and small molecules (ligands), bypassing the need for manual downloads or graphical software switching.
## Workflow & Methodology

# Protein Retrieval:
Connects to the RCSB Protein Data Bank (PDB) REST API to download the raw crystal structure of the target receptor using its specific PDB ID.
# Ligand Batch Processing:
Iterates through a user-defined list of PubChem Compound IDs (CIDs) and fetches their Canonical SMILES strings via the PubChem PUG REST API.
# 3D Structure Generation & Minimization:
-Utilizes RDKit (an open-source cheminformatics toolkit) to accurately convert the 2D SMILES into 3D space.
-Explicit hydrogens are added to simulate physiological relevance.
-3D coordinates are embedded (using a set random seed for reproducibility).
-The geometry is optimized and energy-minimized using the MMFF (Merck Molecular Force Field) to ensure stable conformations for docking.

# Formatting:
 All generated structures are natively exported as .pdb files into a centralized raw_data directory, ensuring immediate compatibility with 3D visualization software (e.g., Biovia Discovery Studio) and subsequent structure preparation tools.

# Dependencies

-requests: For handling API calls to RCSB PDB and PubChem.
-rdkit: For cheminformatics operations3D coordinate embedding, and energy minimization.

### Author:
Abeera_Iftikhar




In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 18.4 MB/s eta 0:00:00


In [3]:
import os
import requests
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import drive

# --- 1. Mount Google Drive ---
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# --- 2. Configuration & Setup ---
PDB_ID = "1HSG"
LIGAND_CIDS = ["5281034", "2244", "3672", "5291", "123631"]

# Create the permanent folders in your Google Drive!
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"
raw_data_dir = os.path.join(BASE_DIR, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# --- 3. Download Protein ---
def download_protein(pdb_id):
    protein_path = os.path.join(raw_data_dir, f"{pdb_id}.pdb")
    print(f"\n--- Fetching Protein {pdb_id} ---")
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    response = requests.get(url)
    if response.status_code == 200:
        with open(protein_path, 'w') as f:
            f.write(response.text)
        print(f"✅ Protein successfully saved to Drive: {protein_path}")
    else:
        print(f"❌ Failed to download protein.")

# --- 4. Generate Clean 3D Ligand as PDB ---
def generate_3d_ligand_pdb(cid):
    ligand_path = os.path.join(raw_data_dir, f"ligand_{cid}_3D.pdb")
    print(f"\n--- Processing Ligand CID {cid} ---")
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/txt"
    response = requests.get(url)

    if response.status_code == 200:
        smiles = response.text.strip()
        print(f"✅ SMILES found: {smiles}")

        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            mol = Chem.AddHs(mol) # Add hydrogens
            AllChem.EmbedMolecule(mol, randomSeed=42) # Generate 3D coordinates
            AllChem.MMFFOptimizeMolecule(mol) # Minimize energy

            # Save directly to Drive as a PDB file
            Chem.MolToPDBFile(mol, ligand_path)
            print(f"✅ 3D Ligand successfully saved to Drive: {ligand_path}")
        else:
            print(f"❌ RDKit could not process SMILES for CID {cid}")
    else:
        print(f"❌ Failed to fetch SMILES from PubChem for CID {cid}.")

# --- Execute ---
# 1. Get the protein
download_protein(PDB_ID)

# 2. Loop through all ligands and process them
for cid in LIGAND_CIDS:
    generate_3d_ligand_pdb(cid)

print("\n--- Notebook 1 Complete ---")
print("All raw files are now permanently saved in your Google Drive under 'Docking_Pipeline/raw_data'!")

Connecting to Google Drive...
Mounted at /content/drive

--- Fetching Protein 1HSG ---
✅ Protein successfully saved to Drive: /content/drive/MyDrive/Docking_Pipeline/raw_data/1HSG.pdb

--- Processing Ligand CID 5281034 ---
✅ SMILES found: CC12CCC3C(C1CCC2(C)O)CCC4C3(CC(=CO)C(=O)C4)C
✅ 3D Ligand successfully saved to Drive: /content/drive/MyDrive/Docking_Pipeline/raw_data/ligand_5281034_3D.pdb

--- Processing Ligand CID 2244 ---
✅ SMILES found: CC(=O)OC1=CC=CC=C1C(=O)O
✅ 3D Ligand successfully saved to Drive: /content/drive/MyDrive/Docking_Pipeline/raw_data/ligand_2244_3D.pdb

--- Processing Ligand CID 3672 ---
✅ SMILES found: CC(C)CC1=CC=C(C=C1)C(C)C(=O)O
✅ 3D Ligand successfully saved to Drive: /content/drive/MyDrive/Docking_Pipeline/raw_data/ligand_3672_3D.pdb

--- Processing Ligand CID 5291 ---
✅ SMILES found: CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5
✅ 3D Ligand successfully saved to Drive: /content/drive/MyDrive/Docking_Pipeline/raw_data/ligand_5291_